In [30]:
%pip install -q groq

Note: you may need to restart the kernel to use updated packages.


In [31]:
import os
os.environ['GROQ_API_KEY'] = "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" # Sendo literal ao video como requisitado

In [32]:
import os
from groq import Groq

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

chat_completion = client.chat.completions.create(
    messages=[
        {"role": "user", "content": "Explain the importance of fast language models"}
    ],
    model="llama-3.3-70b-versatile", # 'llama3-8b-8192' foi descontinuado
    temperature=0 # Acaba com a criatividade do modelo
)

In [33]:
import os 
import groq as Groq
class Agent:
    def __init__(self, client, system):
        self.client = client
        self.system = system
        self.messages = [] # 'Memória' do agente (respostas de cada iteração)
        if self.system is not None: # Se tem um texto prévio a prompt adicionar como instruções na memória
            self.messages.append({
                "role": "system", # Gravar como do sistema
                "content": self.system
            })
    
    def __call__(self, message=""):
        if message:
            self.messages.append({
                "role": "user", # Gravar como usuário (presumidamente vai poder ter uma como tool ou algo do gênero)
                "content": message
            })
        
        result = self.execute()
        self.messages.append({
                "role": "assistant",
                "content": result
            })
        return result

    def execute(self):
        completion = client.chat.completions.create(
            messages=self.messages,
            model="llama-3.3-70b-versatile", # 'llama3-8b-8192' foi descontinuado
            temperature=0
        )
        return completion.choices[0].message.content
    

In [34]:
# Um pouco de engenharia de prompt é essencial para o funcionamento de agentes desse modo, o uso de few-shot garante a estrutura, mas uma estrutura mulitagente ainda é nebulosa, presumidamente cada agente receberia sua prompt própria, mas creio que deve ter um prompt base para todas as prompts de todos agentes nesse contexto
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

search_wikipedia:
e.g. search_wikipedia: Barack
returns the a string with relevant information about Barack

Example session:

Question: When was Barack Obama born?
Thought: I need to find when Barack Obama was born
Action: search_wikipedia: Barack Obama
PAUSE 

You will be called again with this:

Observation: "Barack Hussein Obama II (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. ..."

If you have the answer, output it as the Answer.

Answer: Barack Obama was born in August 4, 1961

Now it's your turn:
""".strip()

In [35]:
# tools (extraído do git, não do vídeo)

# Improvável de ser usado
def calculate(operation: str) -> float:
    return eval(operation)

# Continuarei fazendo em ingles para manter o padrão do vídeo e não adicionar variáveis demais
# Ideia: Fazer uma pesquisa SIMPLES na wikipedia sobre o tema requisitado
# Lib usada: https://pypi.org/project/wikipedia/
%pip install -q wikipedia
import wikipedia

QTD_MAXIMA_WIKIPEDIA = 2500 # Valor completamente subjetivo, mas mandar coisa demais iria confundir o modelo

def search_wikipedia(query:str) -> str:
    try: 
        list_resultados = wikipedia.search(query.lower())

        # debug adicionado por curiosidade
        print('-'*25)
        print(list_resultados) 
        print('-'*25)
        
        # Por ser um exemplo incrivelmente simples vou assumir que o primeiro resultado já é correto
        # Um modelo levemente mais complexo seria usar wikipedia.summary(), mas eu achei meio nebuloso o que (não) vem no retorno dessa função
        # Um modelo melhor, com essa biblioteca, seria mandar essa lista para o modelo e deixar ele decidir qual usar e o quanto (QTD_MAXIMA_WIKIPEDIA) receber
        # Para funcionamento ainda melhor seria usar as outras bibliotecas mencionadas no site ou pesquisar diretamente na internet
        conteudo_pagina = wikipedia.page(list_resultados[0], auto_suggest=False).content
        return conteudo_pagina[:QTD_MAXIMA_WIKIPEDIA]
    except:
        return "Artigo relevante não foi encontrado"

Note: you may need to restart the kernel to use updated packages.


In [36]:
bilbiotecario = Agent(client=client, system=system_prompt)

Usando o ciclo iterativo diretamente

In [37]:
import re
# O maior problema dessa função é o resultado de timeout, se as iterações não forem o bastante ele só desiste e vida que segue, não tem anotação, log, etc
def agent_loop(max_iterations, system, query):
    agent = Agent(client=client, system=system)
    tools = ["calculate", "search_wikipedia"]
    next_prompt = query
    i = 0
    while i < max_iterations:
        i += 1
        result = agent(next_prompt)
        print(result)

        if "PAUSE" in result and "Action" in result:
            action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
            chosen_tool = action[0][0]
            arg = action[0][1]

            if chosen_tool in tools:
                result_tool = eval(f"{chosen_tool}('{arg}')")
                next_prompt = f"Observation: {result_tool}"

            else:
                next_prompt = "Observation: Tool not found"

            print(next_prompt)
            continue

        if "Answer" in result:
            break

In [38]:
agent_loop(max_iterations=5, system=system_prompt, query="When was Albert Einstein born?")

Thought: To find the birth date of Albert Einstein, I should search for information about him on Wikipedia, which is likely to include his birth date.

Action: search_wikipedia: Albert Einstein
PAUSE
-------------------------
['Albert Einstein', 'Hans Albert Einstein', 'Einstein family', 'Albert Brooks', 'Brain of Albert Einstein', 'Political views of Albert Einstein', 'Religious and philosophical views of Albert Einstein', 'List of scientific publications by Albert Einstein', 'Albert Einstein College of Medicine', 'Albert Einstein House']
-------------------------
Observation: Albert Einstein (14 March 1879 – 18 April 1955) was a German-born theoretical physicist best known for developing the theory of relativity. Einstein also made important contributions to quantum theory. His mass–energy equivalence formula E = mc2, which arises from special relativity, has been called "the world's most famous equation". He received the 1921 Nobel Prize in Physics for "his services to theoretical p

In [ ]:
agent_loop(max_iterations=5, system=system_prompt, query="If Albert Einstein was alive today, how old would he be?")
# Tentativa para ver se o modelo vai usar calculate()
# Resultado divertido, modelo foi treinado em 2024 e por isso acha que ainda é 2024, fácil de corrigir de diversos modos (colocar na prompt, adicionar tool para datas, acesso direto a internet, etc), mas eu acho que o resultado direto, demonstrando o limite de um modelo 'cru', é didaticamente mais interessante
# Isso e o fato que o modelo ainda tem a informação da prompt anterior, removendo a necessidade de buscar de novo

Thought: To determine how old Albert Einstein would be if he were alive today, I need to know his birth year and the current year. Albert Einstein was born on March 14, 1879. I can calculate his age by subtracting his birth year from the current year.

Action: calculate: 2024 - 1879

PAUSE
Observation: 145
Thought: Now that I have the result of the calculation, I can determine how old Albert Einstein would be if he were alive today. The calculation shows that he would be 145 years old.

Action: None needed, I have the answer.

Answer: Albert Einstein would be 145 years old if he were alive today.


In [ ]:
agent_loop(max_iterations=5, system=system_prompt, query="Who was Albert Einstein's favorite philosopher?")
# Uma pergunta mais 'díficil' foi feita lendo a wikipedia. Trecho relevante:
# https://en.wikipedia.org/wiki/Albert_Einstein#:~:text=At%20thirteen%2C%20when,%5B27%5D

# Eu honestamente não espero que o modelo consiga responder já que ele pode só pesquisar isso tudo na wikipedia, o que provavelmente não daria certo
# Interessante o modelo acabar pesquisando a mesma coisa da primeira prompt acidentalmente, provavelmente tentando adquirir mais informação, não que tenha ajudado visto o quanto eu limitei a tool

Thought: To answer this question, I need to find information about Albert Einstein's favorite philosopher. This might require searching for biographical information about Einstein or his interests and influences.

Action: search_wikipedia: Albert Einstein 
PAUSE
-------------------------
['Albert Einstein', 'Hans Albert Einstein', 'Einstein family', 'Brain of Albert Einstein', 'Albert Brooks', 'Political views of Albert Einstein', 'Religious and philosophical views of Albert Einstein', 'Albert Einstein College of Medicine', 'List of scientific publications by Albert Einstein', 'Albert Einstein House']
-------------------------
Observation: Albert Einstein (14 March 1879 – 18 April 1955) was a German-born theoretical physicist best known for developing the theory of relativity. Einstein also made important contributions to quantum theory. His mass–energy equivalence formula E = mc2, which arises from special relativity, has been called "the world's most famous equation". He received the

A resposta para 'Who was Albert Einstein's favorite philosopher?' era para ser Kant, ao menos de acordo com o [trecho que eu achei](https://en.wikipedia.org/wiki/Albert_Einstein#:~:text=Kant%20became%20his%20favorite%20philosopher).

Ainda assim o resultado final demonstra a limitação de sistema simples como esse, a minha escolha de limitar a função page com o primeiro resultado da search e, mais importante nessa caso, o quanto de conteúdo o modelo recebe, causou a busca de uma [página que eu não esperava](https://en.wikipedia.org/wiki/Religious_and_philosophical_views_of_Albert_Einstein).
Lendo a página faz sentido a resposta dela, visto que Baruch Spinoza é o filósofo mais mencionado e até mesmo tem trechos dedicados.